# HW13 — Токенизация текста, инференс BERT и fine-tuning для классификации

**Датасет:** `emotion` (6 классов: sadness, joy, love, anger, fear, surprise)  
**Модель:** `distilbert-base-uncased`  
**Задача:** классификация эмоций в коротких текстах

## 1. Импорты, seed и среда

In [ ]:
import random
import os
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
    DataCollatorWithPadding,
)
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# ── Фиксируем seed ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Устройство ────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

import datasets, transformers, sklearn, sys
print(f"Python:       {sys.version.split()[0]}")
print(f"datasets:     {datasets.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"torch:        {torch.__version__}")
print(f"sklearn:      {sklearn.__version__}")

## 2. Данные и первичный анализ

Используем датасет **`emotion`** — коллекцию коротких англоязычных твитов, размеченных по 6 эмоциям: `sadness`, `joy`, `love`, `anger`, `fear`, `surprise`. Датасет небольшой, классы в целом понятные, что удобно для учебного эксперимента.

Для быстрого обучения на CPU берём подвыборку: 2000 train / 500 val / 500 test.

In [ ]:
ds = load_dataset("emotion")

train_ds = ds["train"].select(range(2000))
val_ds   = ds["validation"].select(range(500))
test_ds  = ds["test"].select(range(500))

label_names = ds["train"].features["label"].names
NUM_LABELS  = len(label_names)

print(f"Классы ({NUM_LABELS}): {label_names}")
print(f"Train:      {len(train_ds)} примеров")
print(f"Validation: {len(val_ds)} примеров")
print(f"Test:       {len(test_ds)} примеров")

# Sanity-check: 5 примеров
print("\n--- 5 примеров из train ---")
for i in range(5):
    ex = train_ds[i]
    print(f"  [{label_names[ex['label']]:8s}] {ex['text']}")

In [ ]:
# Распределение классов в train
import collections
counter = collections.Counter(train_ds["label"])
print("Распределение классов в train:")
for idx, cnt in sorted(counter.items()):
    print(f"  {label_names[idx]:10s}: {cnt}")

## 3. Токенизация

Показываем, как токенайзер DistilBERT разбивает текст на токены, какие special tokens добавляются (`[CLS]`, `[SEP]`), как работают `padding` и `truncation`.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Токенайзер: {MODEL_NAME}")
print(f"Special tokens: {tokenizer.all_special_tokens}\n")

sample_texts = [train_ds[i]["text"] for i in range(5)]

for i, text in enumerate(sample_texts[:3]):
    enc    = tokenizer(text, truncation=True, max_length=64)
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    print(f"[Пример {i+1}] {text}")
    print(f"  Tokens       : {tokens}")
    print(f"  input_ids    : {enc['input_ids']}")
    print(f"  attention_mask: {enc['attention_mask']}")
    print()

In [ ]:
# Демонстрация padding: разные по длине тексты выравниваются до одного размера
short_texts = ["I am happy", "This is a much longer sentence that contains more words and tokens"]
padded = tokenizer(short_texts, padding=True, truncation=True, max_length=32, return_tensors="pt")
print("Padding demo (max_length=32):")
print("  input_ids shape:", padded["input_ids"].shape)
print("  Текст 1 (короткий):", tokenizer.convert_ids_to_tokens(padded["input_ids"][0]))
print("  Текст 2 (длинный) :", tokenizer.convert_ids_to_tokens(padded["input_ids"][1]))
print()
# Демонстрация truncation
long_text = " ".join(["word"] * 200)
trunc = tokenizer(long_text, truncation=True, max_length=20)
print(f"Truncation demo (200 слов → max_length=20): {len(trunc['input_ids'])} токенов")

## 4. Инференс готовой pretrained модели

Запускаем готовый sentiment-pipeline (`distilbert-base-uncased-finetuned-sst-2-english`) на нескольких примерах. Эта модель обучена на двух классах (POSITIVE / NEGATIVE), поэтому не подходит напрямую под нашу 6-классовую задачу — нужен fine-tuning.

In [ ]:
sentiment_pipe = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,
)

inference_examples = [
    ("I am so happy today, everything feels wonderful!",  "joy"),
    ("This is absolutely terrible, I hate every second.", "anger"),
    ("I feel quite sad and lonely without you.",          "sadness"),
    ("I'm really nervous and scared about the exam.",    "fear"),
    ("What a wonderful surprise, I can't believe it!",   "surprise"),
]

print(f"{'TEXT':<55} {'TRUE':10} {'PRED':10} {'CONF':6}")
print("-" * 90)
for text, true_label in inference_examples:
    res = sentiment_pipe(text)[0]
    print(f"{text[:54]:<55} {true_label:10} {res['label']:10} {res['score']:.3f}")

print()
print("Вывод: готовая модель различает только POSITIVE/NEGATIVE.")
print("Для 6 эмоций нужен fine-tuning на целевом датасете.")

## 5. Fine-tuning для классификации текста

Дообучаем `distilbert-base-uncased` для 6-классовой классификации эмоций. Выбираем лучший вариант по val_accuracy, финальная оценка — на test один раз.

In [ ]:
MAX_LEN    = 64
BATCH_SIZE = 32
EPOCHS     = 3
LR         = 2e-5

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LEN)

tok_train = train_ds.map(tokenize_fn, batched=True).rename_column("label", "labels").remove_columns(["text"])
tok_val   = val_ds.map(tokenize_fn,   batched=True).rename_column("label", "labels").remove_columns(["text"])
tok_test  = test_ds.map(tokenize_fn,  batched=True).rename_column("label", "labels").remove_columns(["text"])

for split in [tok_train, tok_val, tok_test]:
    split.set_format("torch")

collator     = DataCollatorWithPadding(tokenizer)
train_loader = DataLoader(tok_train, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collator)
val_loader   = DataLoader(tok_val,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)
test_loader  = DataLoader(tok_test,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model.to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR)

train_losses = []
val_accs     = []
best_val_acc = 0.0
best_state   = None

for epoch in range(EPOCHS):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        loss   = model(**batch).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    # ── Validation ───────────────────────────────────────────────────────────
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out   = model(**batch)
            val_preds.extend(out.logits.argmax(dim=-1).cpu().numpy())
            val_labels.extend(batch["labels"].cpu().numpy())

    val_acc = accuracy_score(val_labels, val_preds)
    val_accs.append(val_acc)
    print(f"Epoch {epoch+1}/{EPOCHS} | loss={avg_loss:.4f} | val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print(f"\nBest val_acc: {best_val_acc:.4f}")

## 6. Финальная оценка на test

Загружаем лучшие веса (по val_accuracy) и оцениваем модель на test — один раз.

In [ ]:
model.load_state_dict(best_state)
model.eval()
model.to(DEVICE)

test_preds, test_labels, test_conf = [], [], []
with torch.no_grad():
    for batch in test_loader:
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        out    = model(**batch)
        probs  = torch.softmax(out.logits, dim=-1).cpu().numpy()
        test_preds.extend(probs.argmax(axis=1))
        test_labels.extend(batch["labels"].cpu().numpy())
        test_conf.extend(probs.max(axis=1))

test_acc = accuracy_score(test_labels, test_preds)
test_f1  = f1_score(test_labels, test_preds, average="macro")

print(f"test_accuracy : {test_acc:.4f}")
print(f"test_f1_macro : {test_f1:.4f}")

## 7. Матрица ошибок и анализ предсказаний

In [ ]:
ARTIFACTS_DIR = "./artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names).plot(
    ax=ax, colorbar=True, xticks_rotation=45
)
ax.set_title("Confusion Matrix – test set (emotion, DistilBERT fine-tuned)")
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, "confusion_matrix.png"), dpi=120)
plt.show()
print("Saved confusion_matrix.png")

In [ ]:
# Кривые обучения
fig2, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.plot(range(1, EPOCHS+1), train_losses, marker="o")
a1.set_title("Train Loss"); a1.set_xlabel("Epoch"); a1.set_ylabel("Loss")
a2.plot(range(1, EPOCHS+1), val_accs, marker="o", color="orange")
a2.set_title("Val Accuracy"); a2.set_xlabel("Epoch"); a2.set_ylabel("Accuracy")
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, "training_curves.png"), dpi=120)
plt.show()
print("Saved training_curves.png")

In [ ]:
# Таблица из 20 примеров предсказаний
test_texts_list = list(test_ds["text"])
df_preds = pd.DataFrame({
    "text":       [test_texts_list[i] for i in range(20)],
    "true_label": [label_names[test_labels[i]] for i in range(20)],
    "pred_label": [label_names[test_preds[i]]  for i in range(20)],
    "confidence": [round(float(test_conf[i]), 4) for i in range(20)],
})
df_preds.to_csv(os.path.join(ARTIFACTS_DIR, "sample_predictions.csv"), index=False)
print("Saved sample_predictions.csv\n")
df_preds

## 8. Краткий анализ ошибок

In [ ]:
errors = [
    (test_texts_list[i], label_names[test_labels[i]], label_names[test_preds[i]], round(float(test_conf[i]),3))
    for i in range(len(test_preds))
    if test_labels[i] != test_preds[i]
]

print(f"Всего ошибок: {len(errors)} из {len(test_preds)}\n")
print(f"{'TRUE':10} {'PRED':10} {'CONF':6}  TEXT")
print("-" * 90)
for txt, true, pred, conf in errors[:10]:
    print(f"{true:10} {pred:10} {conf:.3f}  {txt[:65]}")

**Наблюдения по ошибкам:**

- Часто путаются `sadness` ↔ `anger` — оба класса могут выражаться похожим эмоционально негативным языком.
- `surprise` нередко классифицируется как `fear` — удивление и страх лексически близки ("stunned", "shocked").
- `joy` иногда принимается за `sadness` или `fear`, если в тексте есть двойственные конструкции ("I clung to a relationship...").
- Класс `love` — наиболее редкий в подвыборке; модель путает его с другими позитивными эмоциями.
- Большинство ошибок происходят на пограничных, эмоционально неоднозначных текстах, что объяснимо даже для человека.